# Mindestanforderungen 2 - Datenexploration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = Path("output_woche1.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

SO*-Spalten beziehen sich auf StackOverflow-Plattformfunktionen und sind für unsere Recruiting-Fragestellung (Profile, Skills, Präferenzen, Vergütung) nicht relevant.

In [ ]:
so_cols = df.columns[df.columns.str.startswith("SO")]
df = df.drop(columns=so_cols)

print("Dropped SO columns:", len(so_cols))

In [ ]:
df_sorted = df.copy()
df_sorted["YearsCode"] = pd.to_numeric(df_sorted["YearsCode"], errors="coerce")

df_sorted = df_sorted.sort_values("YearsCode", ascending=True)
df_sorted[["YearsCode", "Age", "JobSatisfaction", "WorkExp"]].head(20)


Sortierung nach YearsCode hilft, Erfahrungsstufen zu vergleichen (Junior → Senior) und später HR-Maßnahmen/Angebote zielgruppenspezifisch abzuleiten.

In [ ]:
df_stats = df.copy()
df_stats["YearsCode"] = pd.to_numeric(df_stats["YearsCode"], errors="coerce")

years_code_stats = df_stats["YearsCode"].describe()
median = df_stats["YearsCode"].median()
mode = df_stats["YearsCode"].mode().iloc[0]
iqr = df_stats["YearsCode"].quantile(0.75) - df_stats["YearsCode"].quantile(0.25)

print("Statistische Zusammenfassung - Programmiererfahrung (YearsCode):")
print("----------------------------------------------------")
print(f"Anzahl gültiger Werte: {years_code_stats['count']:.0f}")
print(f"Durchschnitt: {years_code_stats['mean']:.1f} Jahre")
print(f"Median: {median:.1f} Jahre")
print(f"Häufigster Wert (Modus): {mode:.0f} Jahre")
print(f"Standardabweichung: {years_code_stats['std']:.1f} Jahre")
print(f"Minimum: {years_code_stats['min']:.0f} Jahre")
print(f"25%-Quartil: {years_code_stats['25%']:.1f} Jahre")
print(f"75%-Quartil: {years_code_stats['75%']:.1f} Jahre")
print(f"Maximum: {years_code_stats['max']:.0f} Jahre")
print(f"IQR: {iqr:.1f} Jahre")

- RemoteWork ist für Recruiting relevant, da Arbeitsmodelle (Remote/Hybrid/Onsite) mit Präferenzen und ggf. Gehalt/Zufriedenheit zusammenhängen können
- `In-Person` hat hier geringsten salary-median; `Remote` schneidet am besten ab
- die Tabelle zeigt die Mediane (Salary/JobSatisfaction) je Arbeitsmodell im Datensatz – Interpretation daher deskriptiv, nicht kausal.

In [ ]:
df_grp = df.copy()
df_grp["ConvertedCompYearly"] = pd.to_numeric(df_grp["ConvertedCompYearly"], errors="coerce")
df_grp["JobSatisfaction"] = pd.to_numeric(df_grp["JobSatisfaction"], errors="coerce")

grouped = df_grp.groupby("RemoteWork").agg(
    count=("RemoteWork", "count"),
    salary_median=("ConvertedCompYearly", "median"),
    jobsat_median=("JobSatisfaction", "median")
).sort_values("count", ascending=False)

grouped

In [ ]:
df_line = df.copy()
df_line["YearsCode"] = pd.to_numeric(df_line["YearsCode"], errors="coerce")
df_line["ConvertedCompYearly"] = pd.to_numeric(df_line["ConvertedCompYearly"], errors="coerce")

avg_income = df_line.groupby("YearsCode")["ConvertedCompYearly"].mean().dropna()
avg_income = avg_income.sort_index()

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(avg_income.index, avg_income.values)

ax.set_xlabel("Jahre Programmiererfahrung (YearsCode)")
ax.set_ylabel("Durchschnittliches Einkommen (ConvertedCompYearly)")
ax.set_title("Einkommensentwicklung mit Programmiererfahrung")
ax.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()

plt.show()

- Mit zunehmender Erfahrung steigt das durchschnittliche Einkommen tendenziell
- Streuung/Ausreißer sind möglich; Aussage ist aggregiert (Mean) und nicht kausal



In [ ]:
tmp = df[["YearsCode", "ConvertedCompYearly"]].copy()
tmp["YearsCode"] = pd.to_numeric(tmp["YearsCode"], errors="coerce")
tmp["ConvertedCompYearly"] = pd.to_numeric(tmp["ConvertedCompYearly"], errors="coerce")
tmp = tmp.dropna(subset=["YearsCode", "ConvertedCompYearly"])

bins = [0, 2, 5, 10, 15, 25, 50, 100]
tmp["YearsBin"] = pd.cut(tmp["YearsCode"], bins=bins, include_lowest=True)

med_income = tmp.groupby("YearsBin", observed=True)["ConvertedCompYearly"].median()

fig, ax = plt.subplots(figsize=(10,6))
ax.plot(med_income.index.astype(str), med_income.values, marker="o")
ax.set_title("Median Einkommen nach Erfahrungs-Bins")
ax.set_xlabel("YearsCode (Binned)")
ax.set_ylabel("Median ConvertedCompYearly")
ax.grid(True, linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

- Median ist robuster gegen Ausreißer als Mean und somit besser geeignet für Gehaltsband-Argumentation
- Erfahrung wirkt als starker Treiber für Vergütung (aggregiert)

In [ ]:
counts = df["RemoteWork"].value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(10,6))
ax.bar(counts.index.astype(str), counts.values)
ax.set_title("Verteilung RemoteWork")
ax.set_xlabel("Arbeitsmodell")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


- Zeigt, welche Arbeitsmodelle im Datensatz dominieren
- Relevant für Recruiting: Stellenanzeigen sollten Arbeitsmodell klar kommunizieren, weil es stark nachgefragt sein kann

In [ ]:
lang_series = df["LanguageHaveWorkedWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Genutzte Programmiersprachen (HaveWorkedWith)")
ax.set_xlabel("Programmiersprache")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

lang_counts

- Zeigt dominante Skill-Stacks im Markt → hilft beim Formulieren von Must-Have/Nice-to-Have Skills
- Long-Tail sichtbar: viele Nischensprachen, die eher rollen-/domänenspezifisch sind

In [ ]:
lang_series = df["LanguageWantToWorkWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Zukünftig genutzte Programmiersprachen (WantToWorkWith)")
ax.set_xlabel("Programmiersprache")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

lang_counts

- Zeigt Tech-Trends/Präferenzen → hilfreich für Tech-Branding und Zukunfts-Stacks
- Gap Have vs Want kann auf Upskilling-Interesse hinweisen

In [ ]:
lang_series = df["DatabaseWantToWorkWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Zukünftig genutzte Datenbanken")
ax.set_xlabel("Datenbanken")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

lang_counts

In [ ]:
lang_series = df["DatabaseHaveWorkedWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Genutzte Datenbanken (HaveToWorkWith)")
ax.set_xlabel("Datenbanken")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

lang_counts

In [ ]:
lang_series = df["PlatformWantToWorkWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Zukünftig genutzte Plattformen (WantToWorkWith)")
ax.set_xlabel("Plattformen")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
lang_counts

In [ ]:
lang_series = df["PlatformHaveWorkedWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Genutzte Plattformen")
ax.set_xlabel("Plattformen")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
lang_counts

In [ ]:
lang_series = df["WebframeWantToWorkWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Zukünftig genutzte Web-Frameworks (WantToWorkWith)")
ax.set_xlabel("Web-Frameworks")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

lang_counts

In [ ]:
lang_series = df["WebframeHaveWorkedWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Genutzte Web-Frameworks")
ax.set_xlabel("Web-Frameworks")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
lang_counts

In [ ]:
lang_series = df["DevEnvsWantToWorkWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12,6))
ax.bar(lang_counts.index.astype(str), lang_counts.values)
ax.set_title("Top 15: Zukünftig genutzte Entwicklungsumgebung (WantToWorkWith)")
ax.set_xlabel("Entwicklungsumgebungen")
ax.set_ylabel("Anzahl")
ax.grid(True, axis="y", linestyle="--", alpha=0.7)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
lang_counts

In [ ]:
lang_series = df["DevEnvsHaveWorkedWith"].fillna("NaN").astype(str).str.split(";")
lang_exploded = lang_series.explode()

lang_counts = lang_exploded.value_counts().head(15)
